# Red team taxonomy generation

Generates a prohibited-actions taxonomy for the deployed agent and saves the ID for the run notebook.
Source: https://learn.microsoft.com/azure/foundry/how-to/develop/run-ai-red-teaming-cloud

In [ ]:
import json as _json
import os, pathlib, json
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# Discover the active azd env dynamically: prefer AZURE_ENV_NAME, else read
# the 'defaultEnvironment' from agent/.azure/config.json.
AGENT_DIR = pathlib.Path('..') / 'agent'
_azd_base = AGENT_DIR / '.azure'
_env_name = os.environ.get('AZURE_ENV_NAME', '')
if not _env_name:
    _config = _azd_base / 'config.json'
    if _config.exists():
        try:
            _env_name = _json.loads(_config.read_text()).get('defaultEnvironment', '')
        except Exception:
            _env_name = ''
if _env_name and (_azd_base / _env_name / '.env').exists():
    AZD_ENV = _azd_base / _env_name / '.env'
else:
    AZD_ENV = next(
        (p / '.env' for p in sorted(_azd_base.iterdir()) if p.is_dir() and (p / '.env').exists()),
        _azd_base / 'default' / '.env',
    )
if AZD_ENV.exists():
    load_dotenv(AZD_ENV)
load_dotenv()
endpoint = os.environ['AZURE_AI_PROJECT_ENDPOINT']
agent_name = os.environ.get('AZURE_AI_AGENT_NAME', 'agent-framework-agent-basic-responses')
project = AIProjectClient(endpoint=endpoint, credential=DefaultAzureCredential())
# Resolve the latest deployed version. Hardcoding '1' silently yields zero red-team items if newer versions exist.
latest = project.agents.get(agent_name=agent_name)
agent = project.agents.get_version(agent_name=agent_name, agent_version=str(latest.versions['latest'].version))
print('azd env:', _env_name or '(fallback)')
print('agent:  ', agent.name, 'version:', agent.version)

In [ ]:
from azure.ai.projects.models import (
    AzureAIAgentTarget,
    AgentTaxonomyInput,
    EvaluationTaxonomy,
    RiskCategory,
)
target = AzureAIAgentTarget(name=agent_name, version=agent.version)
taxonomy = project.beta.evaluation_taxonomies.create(
    name=f'{agent_name}-prohibited-actions',
    body=EvaluationTaxonomy(
        description='Demo red team taxonomy for prohibited actions',
        taxonomy_input=AgentTaxonomyInput(
            risk_categories=[RiskCategory.PROHIBITED_ACTIONS],
            target=target,
        ),
    ),
)
print('taxonomy id:', taxonomy.id)
TAXONOMY_ID = taxonomy.id
TARGET_DICT = target.as_dict()

In [ ]:
pathlib.Path('../artifacts').mkdir(exist_ok=True)
pathlib.Path('../artifacts/pre-staged-taxonomy.json').write_text(
    json.dumps({'id': TAXONOMY_ID, 'target': TARGET_DICT, 'agent_name': agent_name, 'agent_version': agent.version}, indent=2)
)
print('Saved artifacts/pre-staged-taxonomy.json')